# 04 · CUB labels — what did preprocessing change?

This notebook reads raw CUB image-level attribute annotations before looking at a model.

For image `i` and attribute `j`:

- `a_ij∈{0,1}` is the raw presence annotation;
- `certainty_ij=1` means “not visible” in the official CUB convention;
- `y_i` is species.

The CBM preprocessing replaces image-level `a_ij` with a species-level majority value for the selected 112 attributes. We measure variation before that replacement, repeat after excluding not-visible negative annotations, and reproduce the official majority rule. This identifies a label/visibility concern; it does not by itself prove what a trained model reads.


In [ ]:
import os,sys
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
REPO=Path.cwd().parent
try:sys.path.insert(0,str(REPO/"analysis"));from plotting import set_paper_style;set_paper_style()
except Exception:pass
plt.rcParams["figure.dpi"]=120
cands=[REPO.parent/"data"/"CUB_200_2011",Path(os.environ.get("CURATED_DATA",""))/"CUB_200_2011"]
CUB=next((q for q in cands if (q/"image_class_labels.txt").exists()),None)
assert CUB is not None,f"CUB not found: {cands}"
def load_raw(root):
 rows=[]
 with open(root/"attributes"/"image_attribute_labels.txt") as f:
  for line in f:
   q=line.split()
   if len(q)>=4:rows.append((int(q[0]),int(q[1]),int(q[2]),int(q[3])))
 n_img=max(r[0] for r in rows);n_attr=max(r[1] for r in rows)
 A=np.zeros((n_img,n_attr),np.int8);C=np.zeros((n_img,n_attr),np.int8)
 for i,j,a,c in rows:A[i-1,j-1]=a;C[i-1,j-1]=c
 cls=np.loadtxt(root/"image_class_labels.txt",dtype=int);y=np.zeros(n_img,int);y[cls[:,0]-1]=cls[:,1]
 return A,C,y
A,C,y=load_raw(CUB)
sys.path.insert(0,str(REPO/"external"/"minimal_cbm"));from src.datasets.cub200 import USED_ATTRIBUTES
used=np.array(USED_ATTRIBUTES)-1;A=A[:,used];C=C[:,used]
print(f"{len(A)} images · {len(np.unique(y))} species · {A.shape[1]} selected attributes")


## 1 · Does an attribute vary within a species?

For each species `s` and attribute `j`, calculate `std(a_ij | y_i=s)`. A nonzero value means raw image annotations disagree within that species.

Because some negative annotations mean “part not visible,” we report two versions:

1. all raw annotations;
2. only observations not marked as invisible negatives (`not(a_ij=0 and certainty_ij=1)`).

The second is a stricter test of whether disagreement remains after the clearest visibility artifact is removed.


In [ ]:
def variation(Ax,Cx,yx):
 all_v=[];eligible_v=[]
 for s in np.unique(yx):
  a,c=Ax[yx==s],Cx[yx==s]
  all_v.extend(a.std(0)>0)
  for j in range(a.shape[1]):
   keep=~((a[:,j]==0)&(c[:,j]==1));vals=a[keep,j]
   eligible_v.append((vals.std()>0) if len(vals)>=2 else np.nan)
 return np.mean(all_v),np.nanmean(eligible_v)
keep70=y<=70
rows=[]
for name,k in [("full CUB",np.ones(len(y),bool)),("first 70 classes",keep70)]:
 va,ve=variation(A[k],C[k],y[k]);rows.append(dict(partition=name,all_annotations=va,exclude_invisible_negatives=ve))
V=pd.DataFrame(rows).set_index('partition');display((100*V).round(1))
V.plot.bar(figsize=(8,3.5),color=["#0072B2","#009E73"]);plt.ylabel("% species–attribute pairs with variation")
plt.title("Within-species label variation before majority standardization");plt.xticks(rotation=0);plt.show()


## 2 · Reproduce the official species-majority standardization

For each species and attribute, the preprocessing counts positive and negative annotations, excluding negative annotations with `certainty=1` (“not visible”), then assigns the majority label to every image of that species.

Let `a^MV_sj` be that species-level majority. Count

`flip_ij = 1[a_ij != a^MV_{y_i,j}]`.

This measures how often processed training targets overwrite the raw image annotation. Some flips may correct noisy or invisible annotations; others may erase genuine image variation. The count alone does not decide which.


In [ ]:
def official_majority(Ax,Cx,yx):
 classes=np.unique(yx);mv=np.zeros((len(classes),Ax.shape[1]),np.int8)
 for si,s in enumerate(classes):
  a,c=Ax[yx==s],Cx[yx==s]
  for j in range(a.shape[1]):
   eligible=~((a[:,j]==0)&(c[:,j]==1));vals=a[eligible,j]
   n0=(vals==0).sum();n1=(vals==1).sum();mv[si,j]=1 if n1>=n0 else 0
 mapped=np.vstack([mv[np.where(classes==s)[0][0]] for s in yx])
 return mv,mapped
rows=[]
for name,k in [("full CUB",np.ones(len(y),bool)),("first 70 classes",keep70)]:
 mv,mapped=official_majority(A[k],C[k],y[k]);flips=(A[k]!=mapped)
 rows.append(dict(partition=name,cells=flips.size,flipped=flips.sum(),flip_rate=flips.mean()))
R=pd.DataFrame(rows).set_index('partition');display(R.round(4))
fig,ax=plt.subplots(figsize=(6,3.3));ax.bar(R.index,100*R.flip_rate,color=["#0072B2","#D55E00"])
ax.set_ylabel("% raw image labels overwritten");ax.set_title("Effect of official species-majority preprocessing");plt.show()
MV_FULL,_=official_majority(A,C,y)


## 3 · Are some species–attribute labels genuinely split?

For each species and attribute, calculate prevalence among annotations eligible for the official vote:

`p_sj = mean(a_ij | y_i=s, eligible_ij)`.

Values between 0.3 and 0.7 mean neither label dominates strongly. These cases are harder to dismiss as a single annotation mistake, although viewpoint and visibility can still contribute.


In [ ]:
prev=[]
for s in np.unique(y):
 a,c=A[y==s],C[y==s]
 for j in range(a.shape[1]):
  eligible=~((a[:,j]==0)&(c[:,j]==1));vals=a[eligible,j]
  prev.append(vals.mean() if len(vals) else np.nan)
prev=np.asarray(prev);amb=(prev>=.3)&(prev<=.7)
print(f"eligible species–attribute pairs in 30–70% band: {100*np.nanmean(amb):.1f}%")
fig,ax=plt.subplots(figsize=(7,3.3));ax.hist(prev[np.isfinite(prev)],bins=50,color="#0072B2")
ax.axvspan(.3,.7,color="#D55E00",alpha=.2,label="30–70% band");ax.set_xlabel("eligible within-species prevalence p_sj")
ax.set_ylabel("species–attribute pairs");ax.set_title("Variation remaining in annotations used for majority vote");ax.legend();plt.show()


## What this establishes

The raw data determine three quantities: within-species disagreement, disagreement remaining after the clearest invisible-negative cases are excluded, and the fraction of image labels overwritten by the official species-majority target.

These results motivate the CUB70 mask test. They do not prove backwash: a trained model may still use correct local visual evidence. Notebooks 05 and 06 therefore compare `z_j` and `c_pred_j` with actual part visibility on masked real images.
